##### Same as before (Skip/collapse)

In [1]:
import torch
from torchvision import datasets, transforms

In [2]:
transform = transforms.ToTensor()

In [3]:
train_data = datasets.MNIST(
    root="data",
    train=True,
    download=False,
    transform=transform
)

test_data = datasets.MNIST(
    root="data",
    train=True,
    download=False,
    transform=transform
)

In [4]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

##### Goal:

> Train a CNN on MNIST and see it work.

### Why we can’t use nn.Linear directly

MNIST images come in as:

`[batch, 1, 28, 28]`


nn.Linear expects:

`[batch, features]`


CNNs keep the spatial structure, so we use nn.Conv2d.

### Define the smallest CNN that actually works

In [5]:
import torch.nn as nn

model = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3),  # [1,28,28] → [8,26,26]
    nn.ReLU(),
    nn.MaxPool2d(2),    # [8,26,26] → [8,13,13]
    nn.Flatten(),    # [8,13,13] → [1352]
    nn.Linear(8 * 13 * 13, 10)  # 10 digit classes
)

#### What is nn.Conv2d

Think of it like this:

- An image is just numbers arranged in a grid

- A CNN looks at small squares of the image

- It slides that square over the image and learns patterns

That sliding square is called a kernel (or filter).

`nn.Conv2d(1, 8, 3)`

means:

- 1 → input has 1 channel

    - MNIST is grayscale → not RGB → so 1 channel

- 8 → learn 8 different filters

    - each filter looks for a different pattern (edge, curve, etc.)

- 3 → each filter is 3×3 pixels

#### What is nn.MaxPool2d(2)

`nn.MaxPool2d(2)`

This means:

- Take a 2×2 block

- Keep only the largest value

- Throw the rest away

Effect:

- reduces width & height by half

- keeps strongest features

So:

`[8, 26, 26] → [8, 13, 13]`

#### Why do we use nn.Flatten()

Before this point, data looks like:

`[batch, channels, height, width]`


But nn.Linear expects:

`[batch, features]`


So we flatten:

`8 × 13 × 13 = 1352`


That’s where this comes from:

`nn.Linear(8 * 13 * 13, 10)`


Nothing random — just multiplication.

#### Why 10 in the final layer?

MNIST has digits:

`0 1 2 3 4 5 6 7 8 9`


That's 10 classes.

So the model outputs 10 numbers - one per digit.

### Understand this in ONE glance

| Layer       | What it does                 |
| ----------- | ---------------------------- |
| `Conv2d`    | finds small patterns (edges) |
| `ReLU`      | keeps positives              |
| `MaxPool2d` | shrinks image                |
| `Flatten`   | prepares for classifier      |
| `Linear`    | predicts digit               |


### Loss & optimizer (Classification)

In [6]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Why different loss?
- classification
- labels are 0–9
- CrossEntropyLoss handles this correctly

### Training loop

In [7]:

epochs = 3

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    
    for images, labels in train_loader:
        preds = model(images)
        loss = loss_fn(preds, labels)
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    
        epoch_loss += loss.item()
        
    epoch_loss /= len(train_loader)
    print(f"Epoch: {epoch} | Loss: {epoch_loss:.4f}")
        

Epoch: 0 | Loss: 0.3908
Epoch: 1 | Loss: 0.1333
Epoch: 2 | Loss: 0.0968


### Test accuracy

In [8]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        preds = model(images)
        predicted = preds.argmax(dim=1)
        
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        
accuracy = correct / total
print(accuracy)
        

0.9767166666666667


#### What does argmax do?

argmax means:

> “Give me the index of the largest value”

Example:

    scores = [ -1.2, 0.3, 2.1, -0.5 ]
    argmax → 2


Because 2.1 is the largest.

#### What does dim=1 mean?

shape:

`[batch_size, 10]`


- dim=0 → across batches ❌

- dim=1 → across class scores ✅

So:

`preds.argmax(dim=1)`


Means:

> “For each image, pick the digit with the highest score”

Result shape:

`[batch_size]`


Each value is a predicted digit.

#### (predicted==labels).sum().item()

`(predicted==labels)`

- This returns a True or a False
- But Pytorch handles boolean values as 0 and 1

`sum()`

- This will add true values and increase the count.

`item()`

- Extracts integer/float value from the tensor for arithmetic operations (sum)

#### labels.size(0)

`size(0)`

- returns the batch size